In [0]:
spark.conf.set("fs.azure.account.key.stsalesdataproject.dfs.core.windows.net", "8y8YDrOpJvV5mP6o60pi5CeLqkCKxXxfBuUkya0qv15zELIIYj9VIEQaO6B1MrhPIigcXmCX8o/I+ASt")

In [0]:
dbutils.fs.ls("abfss://datalake@stsalesdataproject.dfs.core.windows.net/staging/")


Out[21]: [FileInfo(path='abfss://datalake@stsalesdataproject.dfs.core.windows.net/staging/sample_Global_Superstore.csv', name='sample_Global_Superstore.csv', size=1297, modificationTime=1745943007000)]

In [0]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("abfss://datalake@stsalesdataproject.dfs.core.windows.net/staging/sample_Global_Superstore.csv")

In [0]:
df.display()

Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
CA-2016-152156,11/8/2016,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
CA-2016-152156,11/8/2016,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,South,FUR-CH-10000454,Furniture,Chairs,Hon Deluxe Fabric Upholstered Stacking Chairs,731.94,3,0.0,219.582
US-2015-108966,10/11/2015,2015-10-18,Standard Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
US-2015-108966,10/11/2015,2015-10-18,Standard Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,West,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
CA-2014-115812,6/9/2014,2014-06-14,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,South,FUR-FU-10001487,Furniture,Furnishings,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164


In [0]:
from pyspark.sql.functions import col

# Drop rows where any field is null
df_cleaned = df.dropna()

In [0]:
from pyspark.sql.functions import col, to_date, when,date_format
from pyspark.sql.types import DoubleType

# Remove null rows
df_cleaned = df.dropna()

# Convert types and create new column
df_typed = df_cleaned.withColumn("Sales", col("Sales").cast(DoubleType())) \
                     .withColumn("Profit", col("Profit").cast(DoubleType())) \
                     .withColumn("Order Date", date_format(col("Order Date"), "MM/dd/yyyy"))

# Add Profit Margin
df_transformed = df_typed.withColumn("Profit Margin", 
                                     when(col("Sales") != 0, col("Profit") / col("Sales")).otherwise(0))

df_transformed.show(5)


+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+
|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|      Profit Margin|
+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+
|CA-2016-152156|      null|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col...|  261.96|

In [0]:
df_transformed .write.mode("overwrite").parquet(curated_path)

In [0]:
display('abfss://datalake@stsalesdataproject.dfs.core.windows.net/curated/globalstore_summary/')

'abfss://datalake@stsalesdataproject.dfs.core.windows.net/curated/globalstore_summary/'